In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Connect to DuckDB database
con = duckdb.connect("olist.duckdb")
print("Connected Successfully!")

Connected Successfully!


In [3]:
#Number of Records

In [4]:
con.execute("""
SELECT COUNT(*) AS total_rows
FROM master_orders
""").fetchdf()

,total_rows
0,119143


In [5]:
#Number of Unique Customers

In [6]:
con.execute("""
SELECT COUNT(DISTINCT customer_unique_id) AS customers
FROM master_orders
""").fetchdf()

,customers
0,96096


In [7]:
#Number of Unique Orders

In [8]:
con.execute("""
SELECT COUNT(DISTINCT order_id) AS orders
FROM master_orders
""").fetchdf()

,orders
0,99441


In [9]:
#Preview the Dataset

In [10]:
master_orders = con.execute("""
SELECT *
FROM master_orders
LIMIT 5
""").fetchdf()

master_orders

,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,product_category_name_english,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,87285b34884572647811a353c7ac498a,housewares,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,2,voucher,1,18.59,4,None,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,595fac2a385ac33a80bd5114aec74eb8,perfumery,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,1,boleto,1,141.46,4,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,aa4383b373c6aca5d8797843e5594415,auto,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,1,credit_card,3,179.12,5,None,None,2018-08-18,2018-08-22 19:07:58
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1,65266b2da20d04dbe00c5c2d3bb7859e,stationery,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,1,credit_card,1,28.62,5,None,None,2018-02-17,2018-02-18 13:02:51
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,1,060cb19345d90064d1015407193c233d,auto,8581055ce74af1daba164fdbd55a40de,2017-07-13 22:10:13,147.90,27.36,1,credit_card,6,175.26,4,None,None,2017-07-27,2017-07-27 22:48:30


In [11]:
#Verify Duplicate Orders

In [12]:
con.execute("""
SELECT
    order_id,
    COUNT(*) AS rows_per_order
FROM master_orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY rows_per_order DESC
LIMIT 10;
""").fetchdf()

,order_id,rows_per_order
0,895ab968e7bb0d5659d16cd74cd1650c,63
1,fedcd9f7ccdc8cba3a18defedd1a5547,38
2,fa65dad1b0e818e3ccc5cb0e39231352,29
3,ccf804e764ed5650cd8759557269dc13,26
4,465c2e1bee4561cb39e0db8c5993aafc,24
5,68986e4324f6a21481df4e6e89abcf01,24
6,c6492b842ac190db807c15aff21a7dd6,24
7,a3725dfe487d359b5be08cac48b64ec5,24
8,6d58638e32674bebee793a47ac4cbadc,24
9,5a3b1c29a49756e75f1ef513383c0c12,22


In [13]:
#Create Order-Level Feature Table

In [14]:
con.execute("""
CREATE OR REPLACE TABLE order_features AS
SELECT
    order_id,
    customer_unique_id,
    MIN(customer_city) AS customer_city,
    MIN(customer_state) AS customer_state,
    MIN(order_purchase_timestamp) AS purchase_date,
    MIN(order_delivered_customer_date) AS delivered_date,
    MIN(order_estimated_delivery_date) AS estimated_delivery,
    ------------------------------------------------------------------
    -- Monetary Features
    ------------------------------------------------------------------
    SUM(price) AS item_value,
    SUM(freight_value) AS freight,
    SUM(price + freight_value) AS order_total,
    MAX(payment_value) AS payment_value,
    ------------------------------------------------------------------
    -- Basket Features
    ------------------------------------------------------------------
    COUNT(product_id) AS total_items,
    COUNT(DISTINCT product_id) AS unique_products,
    COUNT(DISTINCT product_category_name_english)
        AS unique_categories,
    ------------------------------------------------------------------
    -- Review Features
    ------------------------------------------------------------------
    AVG(review_score) AS review_score,
    ------------------------------------------------------------------
    -- Payment Features
    ------------------------------------------------------------------
    MAX(payment_installments) AS installments,
    MAX(payment_type) AS payment_type
FROM master_orders
GROUP BY
    order_id,
    customer_unique_id;
""")
print("order_features created successfully!")

order_features created successfully!


In [15]:
#Inspect Order Features

In [16]:
order_features = con.execute("""
SELECT *
FROM order_features
LIMIT 10
""").fetchdf()
order_features

,order_id,customer_unique_id,customer_city,customer_state,purchase_date,delivered_date,estimated_delivery,item_value,freight,order_total,payment_value,total_items,unique_products,unique_categories,review_score,installments,payment_type
0,432aaf21d85167c2c86ec9448c4e42cc,04cf8185c71090d28baa4407b2e6d600,sao paulo,SP,2018-03-01 14:14:28,2018-03-12 23:36:26,2018-03-21,38.25,16.11,54.36,54.36,1,1,1,4.00,1,credit_card
1,734e7d1bbaeb2ff82521ca0fe6fb6f79,f7603d34c795584792a484186233e6e5,sao paulo,SP,2018-06-11 08:18:19,2018-06-14 21:32:21,2018-07-05,29.99,13.47,43.46,43.46,1,1,1,5.00,1,credit_card
2,d1f54c6a2a7015c5687439d6d4e971ba,6898b2273c4c2e9f697bb597c7df0837,recife,PE,2017-04-27 11:08:59,2017-05-15 12:00:17,2017-06-16,68.00,24.97,92.97,92.97,1,1,1,4.00,1,credit_card
3,e5a7c745f93cb381d78042cdef65b05b,b8a13301c527ebcd83e27cbfb7f06a9a,sao caetano do sul,SP,2017-09-14 18:32:52,2017-09-19 14:42:16,2017-10-02,19.90,11.85,31.75,31.75,1,1,1,5.00,1,boleto
4,d83706c29baf36eedf5e8adfb0da304e,eb9414e9c56dacd436c2b7a274b34983,baixo guandu,ES,2017-03-13 11:36:11,2017-03-27 14:12:27,2017-04-12,27.90,16.05,43.95,43.95,1,1,1,5.00,1,boleto
5,9d513821c0477231fc7c1bfd684d13d8,3abdf4f27efce96f8573a88343a4084e,sao paulo,SP,2017-02-20 21:31:59,2017-03-21 10:35:41,2017-03-17,160.80,31.34,192.14,192.14,2,1,1,5.00,1,credit_card
6,df8e5e994bcc820fcf403f9a875201e6,9107708e80782c0f6ccc2b607f5f2a6c,dracena,SP,2018-08-01 13:30:12,2018-08-06 22:28:21,2018-08-17,75.00,21.17,96.17,96.17,1,1,1,5.00,8,credit_card
7,e995d128a5a5bd59d704a736c4a75fd5,1df745bea08a8a107db186724c8cd2a8,bom jesus dos perdoes,SP,2018-03-10 20:39:19,2018-03-21 20:32:39,2018-03-28,59.90,13.78,73.68,73.68,1,1,1,5.00,7,credit_card
8,e6c3435aca88a50c893f06b6b13e9242,0dd47ea03f3e61825df9e2bc94c292f1,belo horizonte,MG,2017-06-19 18:00:01,2017-06-26 16:21:57,2017-07-10,51.90,12.49,64.39,64.39,1,1,1,5.00,6,credit_card
9,1d5b194c01e762aa59a47ae8dca7199f,00ecc71175078e2cac5547a5656e9622,sao paulo,SP,2017-06-10 18:02:25,2017-06-19 18:33:40,2017-06-28,219.99,15.14,235.13,235.13,1,1,1,5.00,10,credit_card


In [17]:
#Validate Order-Level Table

In [18]:
con.execute("""
SELECT COUNT(*)
FROM order_features
""").fetchdf()

,count_star()
0,99441


In [19]:
#Check Missing Values

In [20]:
order_features.isnull().sum()

order_id              0
customer_unique_id    0
customer_city         0
customer_state        0
purchase_date         0
delivered_date        0
estimated_delivery    0
item_value            0
freight               0
order_total           0
payment_value         0
total_items           0
unique_products       0
unique_categories     0
review_score          0
installments          0
payment_type          0
dtype: int64

In [21]:
#Summary Statistics

In [22]:
order_features.describe()

,purchase_date,delivered_date,estimated_delivery,item_value,freight,order_total,payment_value,total_items,unique_products,unique_categories,review_score,installments
count,10,10,10,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00,10.00
mean,2017-10-06 20:21:28.500000,2017-10-18 03:03:36.500000,2017-10-30 04:48:00,75.16,17.64,92.80,92.80,1.10,1.00,1.00,4.80,3.70
min,2017-02-20 21:31:59,2017-03-21 10:35:41,2017-03-17 00:00:00,19.90,11.85,31.75,31.75,1.00,1.00,1.00,4.00,1.00
25%,2017-05-08 12:52:20.500000,2017-05-24 07:38:37.750000,2017-06-19 00:00:00,32.05,13.55,46.55,46.55,1.00,1.00,1.00,5.00,1.00
50%,2017-08-02 06:16:26.500000,2017-08-08 03:32:06.500000,2017-08-21 00:00:00,55.90,15.60,69.03,69.03,1.00,1.00,1.00,5.00,1.00
75%,2018-03-08 13:03:06.250000,2018-03-19 15:18:35.750000,2018-03-26 06:00:00,73.25,19.91,95.37,95.37,1.00,1.00,1.00,5.00,6.75
max,2018-08-01 13:30:12,2018-08-06 22:28:21,2018-08-17 00:00:00,219.99,31.34,235.13,235.13,2.00,1.00,1.00,5.00,10.00
std,NaN,NaN,NaN,64.83,6.30,67.77,67.77,0.32,0.00,0.00,0.42,3.62


In [23]:
#Create Delivery Features

In [24]:
con.execute("""
ALTER TABLE order_features
ADD COLUMN delivery_days BIGINT;
""")

In [25]:
con.execute("""
UPDATE order_features
SET delivery_days =
DATEDIFF(
'day',
purchase_date,
delivered_date
);
""")

In [26]:
con.execute("""
ALTER TABLE order_features
ADD COLUMN delivery_delay BIGINT;
""")

In [27]:
con.execute("""
UPDATE order_features
SET delivery_delay =
DATEDIFF(
'day',
estimated_delivery,
delivered_date
);
""")

In [28]:
#Step 11: Weekend Purchase Flag

In [29]:
con.execute("""
ALTER TABLE order_features
ADD COLUMN weekend_order INTEGER;
""")

In [30]:
con.execute("""
UPDATE order_features
SET weekend_order =
CASE
WHEN DAYOFWEEK(purchase_date) IN (0,6)
THEN 1
ELSE 0
END;
""")

In [31]:
#Step 12: Inspect Final Order Features

In [32]:
order_features = con.execute("""
SELECT *
FROM order_features
LIMIT 10
""").fetchdf()
order_features

,order_id,customer_unique_id,customer_city,customer_state,purchase_date,delivered_date,estimated_delivery,item_value,freight,order_total,payment_value,total_items,unique_products,unique_categories,review_score,installments,payment_type,delivery_days,delivery_delay,weekend_order
0,432aaf21d85167c2c86ec9448c4e42cc,04cf8185c71090d28baa4407b2e6d600,sao paulo,SP,2018-03-01 14:14:28,2018-03-12 23:36:26,2018-03-21,38.25,16.11,54.36,54.36,1,1,1,4.00,1,credit_card,11,-9,0
1,734e7d1bbaeb2ff82521ca0fe6fb6f79,f7603d34c795584792a484186233e6e5,sao paulo,SP,2018-06-11 08:18:19,2018-06-14 21:32:21,2018-07-05,29.99,13.47,43.46,43.46,1,1,1,5.00,1,credit_card,3,-21,0
2,d1f54c6a2a7015c5687439d6d4e971ba,6898b2273c4c2e9f697bb597c7df0837,recife,PE,2017-04-27 11:08:59,2017-05-15 12:00:17,2017-06-16,68.00,24.97,92.97,92.97,1,1,1,4.00,1,credit_card,18,-32,0
3,e5a7c745f93cb381d78042cdef65b05b,b8a13301c527ebcd83e27cbfb7f06a9a,sao caetano do sul,SP,2017-09-14 18:32:52,2017-09-19 14:42:16,2017-10-02,19.90,11.85,31.75,31.75,1,1,1,5.00,1,boleto,5,-13,0
4,d83706c29baf36eedf5e8adfb0da304e,eb9414e9c56dacd436c2b7a274b34983,baixo guandu,ES,2017-03-13 11:36:11,2017-03-27 14:12:27,2017-04-12,27.90,16.05,43.95,43.95,1,1,1,5.00,1,boleto,14,-16,0
5,9d513821c0477231fc7c1bfd684d13d8,3abdf4f27efce96f8573a88343a4084e,sao paulo,SP,2017-02-20 21:31:59,2017-03-21 10:35:41,2017-03-17,160.80,31.34,192.14,192.14,2,1,1,5.00,1,credit_card,29,4,0
6,df8e5e994bcc820fcf403f9a875201e6,9107708e80782c0f6ccc2b607f5f2a6c,dracena,SP,2018-08-01 13:30:12,2018-08-06 22:28:21,2018-08-17,75.00,21.17,96.17,96.17,1,1,1,5.00,8,credit_card,5,-11,0
7,e995d128a5a5bd59d704a736c4a75fd5,1df745bea08a8a107db186724c8cd2a8,bom jesus dos perdoes,SP,2018-03-10 20:39:19,2018-03-21 20:32:39,2018-03-28,59.90,13.78,73.68,73.68,1,1,1,5.00,7,credit_card,11,-7,1
8,e6c3435aca88a50c893f06b6b13e9242,0dd47ea03f3e61825df9e2bc94c292f1,belo horizonte,MG,2017-06-19 18:00:01,2017-06-26 16:21:57,2017-07-10,51.90,12.49,64.39,64.39,1,1,1,5.00,6,credit_card,7,-14,0
9,1d5b194c01e762aa59a47ae8dca7199f,00ecc71175078e2cac5547a5656e9622,sao paulo,SP,2017-06-10 18:02:25,2017-06-19 18:33:40,2017-06-28,219.99,15.14,235.13,235.13,1,1,1,5.00,10,credit_card,9,-9,1


In [33]:
#Step 13: Create Customer Feature Table

In [34]:
con.execute("""
CREATE OR REPLACE TABLE customer_features AS
SELECT
    customer_unique_id,
    ----------------------------------------------------
    -- Customer Information
    ----------------------------------------------------
    MIN(customer_city) AS customer_city,
    MIN(customer_state) AS customer_state,
    ----------------------------------------------------
    -- Order Metrics
    ----------------------------------------------------
    COUNT(order_id) AS total_orders,
    CASE
        WHEN COUNT(order_id) > 1
        THEN 1
        ELSE 0
    END AS repeat_customer,
    ----------------------------------------------------
    -- Monetary Metrics
    ----------------------------------------------------
    SUM(payment_value) AS total_spent,
    AVG(payment_value) AS average_order_value,
    MEDIAN(payment_value) AS median_order_value,
    MIN(payment_value) AS minimum_order_value,
    MAX(payment_value) AS maximum_order_value,
    STDDEV(payment_value) AS spend_std_dev,
    ----------------------------------------------------
    -- Basket Metrics
    ----------------------------------------------------
    SUM(total_items) AS total_items,
    AVG(total_items) AS average_basket_size,
    MAX(total_items) AS largest_basket,
    SUM(unique_products) AS total_unique_products,
    SUM(unique_categories) AS total_categories,
    ----------------------------------------------------
    -- Freight Metrics
    ----------------------------------------------------
    SUM(freight) AS total_freight,
    AVG(freight) AS average_freight,
    ----------------------------------------------------
    -- Review Metrics
    ----------------------------------------------------
    AVG(review_score) AS average_review,
    MIN(review_score) AS minimum_review,
    MAX(review_score) AS maximum_review,
    STDDEV(review_score) AS review_std_dev,
    ----------------------------------------------------
    -- Payment Metrics
    ----------------------------------------------------
    AVG(installments) AS average_installments,
    MAX(installments) AS maximum_installments,
    COUNT(DISTINCT payment_type)
        AS payment_methods_used,
    ----------------------------------------------------
    -- Delivery Metrics
    ----------------------------------------------------
    AVG(delivery_days) AS average_delivery_days,
    AVG(delivery_delay) AS average_delivery_delay,
    ----------------------------------------------------
    -- Time Metrics
    ----------------------------------------------------
    MIN(purchase_date) AS first_purchase,
    MAX(purchase_date) AS last_purchase,
    DATEDIFF(
        'day',
        MAX(purchase_date),
        (SELECT MAX(purchase_date)
         FROM order_features)
    ) AS recency,
    DATEDIFF(
        'day',
        MIN(purchase_date),
        MAX(purchase_date)
    ) AS customer_lifetime_days,
    DATEDIFF(
        'day',
        MIN(purchase_date),
        (SELECT MAX(purchase_date)
         FROM order_features)
    ) AS customer_age,
    ----------------------------------------------------
    -- Weekend Behaviour
    ----------------------------------------------------
    AVG(weekend_order) * 100
        AS weekend_purchase_percentage
FROM order_features
GROUP BY customer_unique_id;
""")
print("customer_features created successfully!")

customer_features created successfully!


In [35]:
#Create Derived Features

In [36]:
#Average Item Price

In [37]:
con.execute("""
ALTER TABLE customer_features
ADD COLUMN average_item_price DOUBLE;
""")
con.execute("""
UPDATE customer_features
SET average_item_price =
total_spent /
NULLIF(total_items,0);
""")

In [38]:
#Freight Percentage

In [39]:
con.execute("""
ALTER TABLE customer_features
ADD COLUMN freight_percentage DOUBLE;
""")
con.execute("""
UPDATE customer_features
SET freight_percentage =
100 * total_freight /
NULLIF(total_spent,0);
""")

In [40]:
#Purchase Frequency

In [41]:
con.execute("""
ALTER TABLE customer_features
ADD COLUMN purchase_frequency DOUBLE;
""")
con.execute("""
UPDATE customer_features
SET purchase_frequency =
total_orders /
NULLIF(customer_lifetime_days,1);
""")

In [42]:
#Category Diversity Ratio

In [43]:
con.execute("""
ALTER TABLE customer_features
ADD COLUMN category_diversity_ratio DOUBLE;
""")
con.execute("""
UPDATE customer_features
SET category_diversity_ratio =
total_categories /
NULLIF(total_items,0);
""")

In [44]:
#Create Review Counts

In [45]:
#Low review customer

In [46]:
con.execute("""
ALTER TABLE customer_features
ADD COLUMN low_review_orders BIGINT;
""")

In [47]:
con.execute("""
UPDATE customer_features c
SET low_review_orders = (
SELECT COUNT(*)
FROM order_features o
WHERE
o.customer_unique_id=c.customer_unique_id
AND review_score<=2
);
""")

In [48]:
#High review customers.

In [49]:
con.execute("""
ALTER TABLE customer_features
ADD COLUMN high_review_orders BIGINT;
""")

In [50]:
con.execute("""
UPDATE customer_features c
SET high_review_orders=(
SELECT COUNT(*)
FROM order_features o
WHERE
o.customer_unique_id=c.customer_unique_id
AND review_score>=4
);
""")

In [51]:
#Preview

In [52]:
customer_features = con.execute("""
SELECT *
FROM customer_features
""").fetchdf()
customer_features.head()

,customer_unique_id,customer_city,customer_state,total_orders,repeat_customer,total_spent,average_order_value,median_order_value,minimum_order_value,maximum_order_value,spend_std_dev,total_items,average_basket_size,largest_basket,total_unique_products,total_categories,total_freight,average_freight,average_review,minimum_review,maximum_review,review_std_dev,average_installments,maximum_installments,payment_methods_used,average_delivery_days,average_delivery_delay,first_purchase,last_purchase,recency,customer_lifetime_days,customer_age,weekend_purchase_percentage,average_item_price,freight_percentage,purchase_frequency,category_diversity_ratio,low_review_orders,high_review_orders
0,df3bcbcb88d40ad1996ad1325fe8f481,praia grande,SP,1,0,160.28,160.28,160.28,160.28,160.28,NaN,2.00,2.00,2,1.00,1.00,23.00,23.00,4.00,4.00,4.00,NaN,1.00,1,1,7.00,-6.00,2018-06-15 15:52:21,2018-06-15 15:52:21,124,0,124,0.00,80.14,14.35,inf,0.50,0,1
1,58704dbb44acbec0448a215ac2871320,sao paulo,SP,1,0,70.72,70.72,70.72,70.72,70.72,NaN,1.00,1.00,1,1.00,1.00,11.73,11.73,5.00,5.00,5.00,NaN,1.00,1,1,6.00,-11.00,2017-10-03 19:29:06,2017-10-03 19:29:06,379,0,379,0.00,70.72,16.59,inf,1.00,0,1
2,e06358b52b33afa8edfe5a0664cfe9c5,ponte serrada,SC,1,0,60.10,60.10,60.10,60.10,60.10,NaN,1.00,1.00,1,1.00,1.00,15.10,15.10,1.00,1.00,1.00,NaN,3.00,3,1,17.00,-8.00,2018-02-19 12:23:34,2018-02-19 12:23:34,240,0,240,0.00,60.10,25.12,inf,1.00,1,0
3,ee66c72bc9c41ae9de8260314b8088f4,juiz de fora,MG,1,0,247.36,247.36,247.36,247.36,247.36,NaN,2.00,2.00,2,1.00,1.00,47.38,47.38,5.00,5.00,5.00,NaN,2.00,2,1,6.00,-16.00,2018-06-20 19:21:04,2018-06-20 19:21:04,119,0,119,0.00,123.68,19.15,inf,0.50,0,1
4,c2e79de2c0a7751e26564a3917f9e86d,sao paulo,SP,1,0,115.44,115.44,115.44,115.44,115.44,NaN,1.00,1.00,1,1.00,1.00,15.45,15.45,5.00,5.00,5.00,NaN,1.00,1,1,7.00,-16.00,2017-05-31 15:39:51,2017-05-31 15:39:51,504,0,504,0.00,115.44,13.38,inf,1.00,0,1


In [53]:
#Step 17:Check Shape

In [54]:
customer_features.shape

(96096, 39)

In [55]:
customer_features.columns

Index(['customer_unique_id', 'customer_city', 'customer_state', 'total_orders',
       'repeat_customer', 'total_spent', 'average_order_value',
       'median_order_value', 'minimum_order_value', 'maximum_order_value',
       'spend_std_dev', 'total_items', 'average_basket_size', 'largest_basket',
       'total_unique_products', 'total_categories', 'total_freight',
       'average_freight', 'average_review', 'minimum_review', 'maximum_review',
       'review_std_dev', 'average_installments', 'maximum_installments',
       'payment_methods_used', 'average_delivery_days',
       'average_delivery_delay', 'first_purchase', 'last_purchase', 'recency',
       'customer_lifetime_days', 'customer_age', 'weekend_purchase_percentage',
       'average_item_price', 'freight_percentage', 'purchase_frequency',
       'category_diversity_ratio', 'low_review_orders', 'high_review_orders'],
      dtype='object')

In [56]:
customer_features.isnull().sum()

customer_unique_id                 0
customer_city                      0
customer_state                     0
total_orders                       0
repeat_customer                    0
total_spent                        1
average_order_value                1
median_order_value                 1
minimum_order_value                1
maximum_order_value                1
spend_std_dev                  93099
total_items                        0
average_basket_size                0
largest_basket                     0
total_unique_products              0
total_categories                   0
total_freight                    676
average_freight                  676
average_review                   716
minimum_review                   716
maximum_review                   716
review_std_dev                 93145
average_installments               1
maximum_installments               1
payment_methods_used               0
average_delivery_days           2740
average_delivery_delay          2740
f

In [57]:
#Step 20: Summary Statistics
customer_features.describe()

C:\Users\swath\anaconda3\lib\site-packages\numpy\lib\function_base.py:4573: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


,total_orders,repeat_customer,total_spent,average_order_value,median_order_value,minimum_order_value,maximum_order_value,spend_std_dev,total_items,average_basket_size,largest_basket,total_unique_products,total_categories,total_freight,average_freight,average_review,minimum_review,maximum_review,review_std_dev,average_installments,maximum_installments,payment_methods_used,average_delivery_days,average_delivery_delay,first_purchase,last_purchase,recency,customer_lifetime_days,customer_age,weekend_purchase_percentage,average_item_price,freight_percentage,purchase_frequency,category_diversity_ratio,low_review_orders,high_review_orders
count,96096.00,96096.00,96095.00,96095.00,96095.00,96095.00,96095.00,2997.00,96096.00,96096.00,96096.00,96096.00,96096.00,95420.00,95420.00,95380.00,95380.00,95380.00,2951.00,96095.00,96095.00,96096.00,93356.00,93356.00,96096,96096,96096.00,96096.00,96096.00,96096.00,95419.00,95419.00,96050.00,95420.00,96096.00,96096.00
mean,1.03,0.03,165.02,159.89,159.85,158.43,161.39,64.58,1.23,1.18,1.19,1.07,1.02,24.84,23.95,4.08,4.07,4.10,0.50,2.92,2.94,1.00,12.51,-11.85,2017-12-30 19:19:10.429206,2018-01-02 12:40:19.655865,288.11,2.72,290.83,22.97,143.70,23.56,inf,0.92,0.15,0.79
min,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,-147.00,2016-09-04 21:15:19,2016-09-04 21:15:19,0.00,0.00,0.00,0.00,0.19,0.00,0.00,0.00,0.00,0.00
25%,1.00,0.00,62.33,61.64,61.62,60.42,61.83,8.32,1.00,1.00,1.00,1.00,1.00,14.10,14.10,4.00,4.00,4.00,0.00,1.00,1.00,1.00,7.00,-17.00,2017-09-11 19:52:06,2017-09-15 09:04:17.250000,164.00,0.00,166.00,0.00,55.59,11.77,NaN,1.00,0.00,1.00
50%,1.00,0.00,106.90,104.70,104.63,103.07,105.38,31.61,1.00,1.00,1.00,1.00,1.00,17.70,17.60,5.00,5.00,5.00,0.00,2.00,2.00,1.00,10.00,-12.00,2018-01-18 13:33:08,2018-01-21 19:39:16,269.00,0.00,272.00,0.00,94.22,18.66,NaN,1.00,0.00,1.00
75%,1.00,0.00,181.86,175.91,175.87,174.36,177.56,75.55,1.00,1.00,1.00,1.00,1.00,26.65,25.63,5.00,5.00,5.00,0.71,4.00,4.00,1.00,16.00,-7.00,2018-05-04 10:38:45,2018-05-06 20:14:49.750000,397.00,0.00,401.00,0.00,161.04,28.32,NaN,1.00,0.00,1.00
max,17.00,1.00,13664.08,13664.08,13664.08,13664.08,13664.08,1646.89,75.00,38.00,63.00,16.00,15.00,1897.76,1897.76,5.00,5.00,5.00,2.83,24.00,24.00,3.00,210.00,188.00,2018-10-17 17:30:18,2018-10-17 17:30:18,773.00,633.00,773.00,100.00,9553.02,6405.41,inf,1.00,3.00,16.00
std,0.21,0.17,229.83,220.71,220.72,220.25,222.40,110.75,0.86,0.72,0.76,0.34,0.27,28.04,26.23,1.34,1.35,1.34,0.82,2.69,2.72,0.07,9.56,10.14,NaN,NaN,153.42,25.45,153.91,41.85,198.26,50.64,NaN,0.21,0.36,0.47


In [58]:
customer_features = con.execute("""
SELECT *
FROM customer_features
""").fetchdf()

In [59]:
customer_features

,customer_unique_id,customer_city,customer_state,total_orders,repeat_customer,total_spent,average_order_value,median_order_value,minimum_order_value,maximum_order_value,spend_std_dev,total_items,average_basket_size,largest_basket,total_unique_products,total_categories,total_freight,average_freight,average_review,minimum_review,maximum_review,review_std_dev,average_installments,maximum_installments,payment_methods_used,average_delivery_days,average_delivery_delay,first_purchase,last_purchase,recency,customer_lifetime_days,customer_age,weekend_purchase_percentage,average_item_price,freight_percentage,purchase_frequency,category_diversity_ratio,low_review_orders,high_review_orders
0,df3bcbcb88d40ad1996ad1325fe8f481,praia grande,SP,1,0,160.28,160.28,160.28,160.28,160.28,NaN,2.00,2.00,2,1.00,1.00,23.00,23.00,4.00,4.00,4.00,NaN,1.00,1,1,7.00,-6.00,2018-06-15 15:52:21,2018-06-15 15:52:21,124,0,124,0.00,80.14,14.35,inf,0.50,0,1
1,58704dbb44acbec0448a215ac2871320,sao paulo,SP,1,0,70.72,70.72,70.72,70.72,70.72,NaN,1.00,1.00,1,1.00,1.00,11.73,11.73,5.00,5.00,5.00,NaN,1.00,1,1,6.00,-11.00,2017-10-03 19:29:06,2017-10-03 19:29:06,379,0,379,0.00,70.72,16.59,inf,1.00,0,1
2,e06358b52b33afa8edfe5a0664cfe9c5,ponte serrada,SC,1,0,60.10,60.10,60.10,60.10,60.10,NaN,1.00,1.00,1,1.00,1.00,15.10,15.10,1.00,1.00,1.00,NaN,3.00,3,1,17.00,-8.00,2018-02-19 12:23:34,2018-02-19 12:23:34,240,0,240,0.00,60.10,25.12,inf,1.00,1,0
3,ee66c72bc9c41ae9de8260314b8088f4,juiz de fora,MG,1,0,247.36,247.36,247.36,247.36,247.36,NaN,2.00,2.00,2,1.00,1.00,47.38,47.38,5.00,5.00,5.00,NaN,2.00,2,1,6.00,-16.00,2018-06-20 19:21:04,2018-06-20 19:21:04,119,0,119,0.00,123.68,19.15,inf,0.50,0,1
4,c2e79de2c0a7751e26564a3917f9e86d,sao paulo,SP,1,0,115.44,115.44,115.44,115.44,115.44,NaN,1.00,1.00,1,1.00,1.00,15.45,15.45,5.00,5.00,5.00,NaN,1.00,1,1,7.00,-16.00,2017-05-31 15:39:51,2017-05-31 15:39:51,504,0,504,0.00,115.44,13.38,inf,1.00,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,7122b4f327033b35961170342401066c,valinhos,SP,1,0,21.45,21.45,21.45,21.45,21.45,NaN,1.00,1.00,1,1.00,1.00,10.96,10.96,5.00,5.00,5.00,NaN,1.00,1,1,6.00,-33.00,2017-01-06 21:30:38,2017-01-06 21:30:38,649,0,649,0.00,21.45,51.10,inf,1.00,0,1
96092,a354ee7ba147e604d8b6c07ab4646a14,passos,MG,1,0,64.09,64.09,64.09,64.09,64.09,NaN,1.00,1.00,1,1.00,1.00,15.10,15.10,5.00,5.00,5.00,NaN,1.00,1,1,9.00,-14.00,2017-05-29 12:58:03,2017-05-29 12:58:03,506,0,506,0.00,64.09,23.56,inf,1.00,0,1
96093,dd38439b7234a962580852c20d68a328,vitoria,ES,1,0,58.95,58.95,58.95,58.95,58.95,NaN,1.00,1.00,1,1.00,1.00,16.05,16.05,5.00,5.00,5.00,NaN,4.00,4,1,8.00,-20.00,2017-03-23 20:06:13,2017-03-23 20:06:13,573,0,573,0.00,58.95,27.23,inf,1.00,0,1
96094,e0b954de94def9406ed69dcf429442c0,sao paulo,SP,1,0,40.17,40.17,40.17,40.17,40.17,NaN,1.00,1.00,1,1.00,1.00,15.27,15.27,4.00,4.00,4.00,NaN,1.00,1,1,6.00,-13.00,2018-07-06 11:18:39,2018-07-06 11:18:39,103,0,103,0.00,40.17,38.01,inf,1.00,0,1


In [60]:
customer_features.to_csv("customer_features.csv", index=False)